# 读取数据

In [1]:
import numpy as np
import pandas as pd
import ast
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.cluster import KMeans
from scipy.ndimage import correlate
import warnings
warnings.filterwarnings("ignore", category=UserWarning) # 屏蔽单维 KMeans 的内存警告

In [2]:
import polars as pl

# Polars 原生支持 parquet，性能极高
try:
    df_total = pl.read_parquet("epv_events.parquet.gzip")
    print(df_total.head())
except Exception as e:
    print(f"读取失败: {e}")

shape: (5, 23)
┌───────┬────────────┬────────────┬────────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ index ┆ _id        ┆ match_id   ┆ event_type ┆ … ┆ stat_poss ┆ destinati ┆ opponent_ ┆ stat_play │
│ ---   ┆ ---        ┆ ---        ┆ ---        ┆   ┆ team_arra ┆ on_player ┆ team      ┆ er_id_arr │
│ i64   ┆ str        ┆ str        ┆ str        ┆   ┆ y         ┆ ---       ┆ ---       ┆ ay        │
│       ┆            ┆            ┆            ┆   ┆ ---       ┆ f64       ┆ f64       ┆ ---       │
│       ┆            ┆            ┆            ┆   ┆ binary    ┆           ┆           ┆ binary    │
╞═══════╪════════════╪════════════╪════════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 0     ┆ 6839d295b7 ┆ 68397b9914 ┆ Mantener   ┆ … ┆ b"[false, ┆ null      ┆ 181.0     ┆ b"[80906, │
│       ┆ 91492b0852 ┆ ffd9f06b06 ┆ el Balon   ┆   ┆ false,fal ┆           ┆           ┆ 105571,15 │
│       ┆ 563c       ┆ a136       ┆            ┆   ┆ se,false, ┆           ┆

In [3]:
df_total.head()


index,_id,match_id,event_type,start_frame,end_frame,origin_player,team,success,norm_origin_pos_x,norm_origin_pos_y,norm_destination_pos_x,norm_destination_pos_y,stat_full_minute,stat_half,stat_posx_array,stat_posy_array,stat_velx_array,stat_vely_array,stat_possteam_array,destination_player,opponent_team,stat_player_id_array
i64,str,str,str,i64,i64,i64,i64,str,f64,f64,f64,f64,f64,f64,binary,binary,binary,binary,binary,f64,f64,binary
0,"""6839d295b791492b0852563c""","""68397b9914ffd9f06b06a136""","""Mantener el Balon""",1782534,1782559,88952,407,"""Fallido""",0.39221,0.21352,0.42538,0.20223,58.0,2.0,"b""[0.36805999999999994,0.59049,0.59078,0.4027599999999999,0.48""…","b""[0.36700999999999995,0.4849899999999999,0.69896,0.4964299999""…","b""[0.03536000000000006,0.017020000000000035,0.0118800000000000""…","b""[-0.03922000000000003,-0.011289999999999911,-0.0169400000000""…","b""[false,false,false,false,false,false,false,false,false,false""…",null,181.0,"b""[80906,105571,152723,156293,169007,170116,212873,227164,4374""…"
1,"""6839d295b13a38a4c952563c""","""68397b9914ffd9f06b06a136""","""Mantener el Balon""",1699658,1699683,88952,407,"""Fallido""",0.77937,0.30743,0.77101,0.22883,20.0,1.0,"b""[0.63099,0.8758600000000001,0.8510500000000001,0.52899000000""…","b""[0.51263,0.45765000000000006,0.51961,0.50654,0.3689500000000""…","b""[0.006650000000000045,-0.016349999999999976,-0.0084600000000""…","b""[-0.03223999999999999,-0.004310000000000036,-0.0005899999999""…","b""[false,false,false,false,false,false,false,false,false,false""…",null,181.0,"b""[80906,105571,152723,156293,169007,170116,212873,227164,4374""…"
2,"""6839d295b791492b0852563d""","""68397b9914ffd9f06b06a136""","""Mantener el Balon""",1782559,1782584,88952,407,"""Fallido""",0.42538,0.20223,0.46825,0.19435,58.0,2.0,"b""[0.40342,0.60751,0.60266,0.42072,0.5109299999999999,0.61968,""…","b""[0.3277899999999999,0.4737,0.68202,0.4933099999999999,0.6857""…","b""[0.040690000000000004,0.019870000000000054,0.018159999999999""…","b""[-0.03045999999999993,-0.01426000000000005,-0.02005999999999""…","b""[false,false,false,false,false,false,false,false,false,false""…",null,181.0,"b""[80906,105571,152723,156293,169007,170116,212873,227164,4374""…"
3,"""6839d295b13a38a4c952563d""","""68397b9914ffd9f06b06a136""","""Mantener el Balon""",1699683,1699708,88952,407,"""Fallido""",0.77101,0.22883,0.79373,0.15423,20.0,1.0,"b""[0.6376400000000001,0.8595100000000001,0.8425900000000001,0.""…","b""[0.48039000000000004,0.45334,0.51902,0.46107000000000004,0.3""…","b""[0.016349999999999976,0.005979999999999985,0.013109999999999""…","b""[-0.01917000000000002,0.02124999999999999,0.0020799999999999""…","b""[false,false,false,false,false,false,false,false,false,false""…",null,181.0,"b""[80906,105571,152723,156293,169007,170116,212873,227164,4374""…"
4,"""6839d295b13a38a4c952563e""","""68397b9914ffd9f06b06a136""","""Mantener el Balon""",1699708,1699733,88952,407,"""Fallido""",0.79373,0.15423,0.82538,0.13314,20.0,1.0,"b""[0.6539900000000001,0.8654900000000001,0.8557,0.533940000000""…","b""[0.46122,0.47459,0.5211,0.43819,0.33180000000000004,0.320650""…","b""[0.02071999999999996,0.019869999999999943,0.0214900000000000""…","b""[-0.01158999999999999,0.0050500000000000544,0.01263000000000""…","b""[false,false,false,false,false,false,false,false,false,false""…",null,181.0,"b""[80906,105571,152723,156293,169007,170116,212873,227164,4374""…"


In [4]:
# 返回一个 Polars Series
unique_matches = df_total['match_id'].unique()

# 转换为列表
match_id_list = df_total['match_id'].unique().to_list()
print(match_id_list)
# 统计独特 match_id 的数量
match_count = df_total.select(pl.col('match_id').n_unique()).item()
print(f"总比赛场数: {match_count}")

['683e6de44fe78cbe3a3415f6', '683e97ac4fe78cbe3aa36866', '683e434f4fe78cbe3ac2b456', '683dfb554fe78cbe3a03d2c9', '683a182b7773b55f2b41f8da', '683a19fe7773b55f2b46f1c0', '683e80834fe78cbe3a65aa86', '683e00964fe78cbe3a11d1b2', '683e426d4fe78cbe3ac061c6', '683df9854fe78cbe3afef5a9', '683e84054fe78cbe3a6ef18f', '683e20674fe78cbe3a66b6cc', '683d8f2d4fe78cbe3aea4da4', '683e61924fe78cbe3a133345', '683d97064fe78cbe3afd467a', '683e2bd84fe78cbe3a854a0a', '683a33037773b55f2b894929', '683e1c094fe78cbe3a5b0903', '683df18f4fe78cbe3ae9e7e7', '683dc8584fe78cbe3a7d131c', '683e535c4fe78cbe3aed37e4', '683e5fc84fe78cbe3a0e7641', '683e64364fe78cbe3a1a43a9', '683e93504fe78cbe3a97b07b', '683dd5ee4fe78cbe3aa07f8e', '683e8cea4fe78cbe3a86a6a1', '683de4534fe78cbe3ac697a9', '683df8a94fe78cbe3afcae64', '683a466d1ffcc57e43690be5', '683d9e964fe78cbe3a1024db', '683dcdd24fe78cbe3a8b44f0', '683dcbfc4fe78cbe3a867858', '683dafb94fe78cbe3a3d676b', '683e23f04fe78cbe3a70090f', '683a50eb1ffcc57e4383b119', '683da1554fe78cbe3a

In [5]:

# 1. 一键切分为字典，并通过 lambda 表达式在切分的同时完成内部排序
match_dict = {
    match_id: sub_df.sort("start_frame")
    for match_id, sub_df in df_total.partition_by("match_id", as_dict=True).items()
}


# 2. 随时通过具体的 match_id 获取对应比赛按时间排好序的事件链

# 陷阱：Polars 的 as_dict=True 默认生成的 Key 是一个元组，如 ('68397b9914ffd9f06b06a136',)
target_match = ("68397b9914ffd9f06b06a136",)

if target_match in match_dict:
    game_df = match_dict[target_match]
    print("成功通过元组 Key 找到比赛！")
if target_match in match_dict:
    game_df = match_dict[target_match]
    print(f"成功获取比赛 {target_match}，其前5个事件：")
    # 临时将显示列数、行数、列宽、字符长度都设为无限制（或极大值）
    with pl.Config(
        tbl_cols=-1,       # -1 代表显示所有列，不再出现中间的 ...
        tbl_rows=20,       # 显式输出的行数（默认是 8 行）
        tbl_width_chars=1000, # 允许单行最宽的字符数，防止列换行错位
        fmt_str_lengths=100,  # 允许字符串/二进制内容显示的最大长度，防止出现 fal...
        ):
        print(game_df.head(5))
else:
    print(f"未在数据集中找到 match_id 为 {target_match} 的比赛。")

成功通过元组 Key 找到比赛！
成功获取比赛 ('68397b9914ffd9f06b06a136',)，其前5个事件：
shape: (5, 23)
┌───────┬──────────────────────────┬──────────────────────────┬────────────┬─────────────┬───────────┬───────────────┬──────┬─────────┬───────────────────┬───────────────────┬────────────────────────┬────────────────────────┬──────────────────┬───────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────┬───────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ ind

In [6]:
df_total["team"].unique()

team
i64
173
174
175
176
177
…
449
450
957


In [7]:
import polars as pl

# 按照 team 分组，统计 match_id 的唯一值数量，并将新列命名为 unique_match_count
unique_matches_df = df_total.group_by('team').agg(
    pl.col('match_id').n_unique().alias('unique_match_count')
)
with pl.Config(tbl_rows=-1, tbl_cols=-1):

    print(unique_matches_df)

shape: (20, 2)
┌──────┬────────────────────┐
│ team ┆ unique_match_count │
│ ---  ┆ ---                │
│ i64  ┆ u32                │
╞══════╪════════════════════╡
│ 179  ┆ 38                 │
│ 176  ┆ 38                 │
│ 1450 ┆ 38                 │
│ 173  ┆ 38                 │
│ 185  ┆ 37                 │
│ 191  ┆ 38                 │
│ 450  ┆ 38                 │
│ 188  ┆ 38                 │
│ 186  ┆ 38                 │
│ 957  ┆ 38                 │
│ 174  ┆ 38                 │
│ 192  ┆ 38                 │
│ 177  ┆ 38                 │
│ 407  ┆ 38                 │
│ 2893 ┆ 37                 │
│ 175  ┆ 38                 │
│ 178  ┆ 38                 │
│ 181  ┆ 38                 │
│ 449  ┆ 38                 │
│ 184  ┆ 38                 │
└──────┴────────────────────┘


In [8]:
import random
import polars as pl

# ==============================================================================
# 构建 10 场比赛的微型测试集 (Test Sandbox)
# ==============================================================================

# 获取所有比赛的 keys (注意：Polars partition_by as_dict=True 的 key 是 tuple 形式)
all_match_keys = list(match_dict.keys())

# 设定固定的随机种子，保证每次运行 Notebook 抽取的都是这 10 场比赛
# 这对于我们在开发过程中对比代码修改前后 (如 5x5网格 vs 1x1网格) 的结果至关重要
random.seed(42) 

# 策略 A：科学随机抽样 10 场比赛（推荐，能覆盖不同球队的战术与阵型特征）
sampled_keys = random.sample(all_match_keys, 30)

# 策略 B：如果你想专门研究某几场特定的焦点战，可以直接在这里硬编码 match_id 元组
# sampled_keys = [('68397b9914ffd9f06b06a136',), ('另一种特定的match_id',), ...]

print("🎯 成功抽取 10 场测试比赛 ID:")
for i, k in enumerate(sampled_keys):
    print(f"  {i+1}. {k[0]}")

# 将这 10 场比赛的数据块重新垂直拼接 (Concat) 成一个新的测试 DataFrame
df = pl.concat([match_dict[k] for k in sampled_keys])

print(f"\n📊 测试集数据规模: {df.shape[0]} 行")

# 确保时间序列严格排序，防止拼接后出现帧数跳跃
df = df.sort(["match_id", "team", "start_frame"])

# ==============================================================================
# 现在，你可以安全地将 test_df 送入方案一的 PC 算力引擎中进行小规模验证了
# df_result = test_df.with_columns(...)
# ==============================================================================

🎯 成功抽取 10 场测试比赛 ID:
  1. 683e71644fe78cbe3a3d71c3
  2. 683a49441ffcc57e43702c17
  3. 683a13af7773b55f2b35fc9e
  4. 683dcb084fe78cbe3a842503
  5. 683dbd814fe78cbe3a60d5e0
  6. 683db33f4fe78cbe3a46bc8c
  7. 683d8b604fe78cbe3ae1298d
  8. 683e9dea4fe78cbe3ab3d44c
  9. 683a44981ffcc57e43644231
  10. 683e82404fe78cbe3a6a42c5
  11. 683e46d34fe78cbe3acc203b
  12. 683a305c7773b55f2b823ea1
  13. 683e5b474fe78cbe3a02652e
  14. 683e0ec54fe78cbe3a37ab86
  15. 683a173b7773b55f2b3f737e
  16. 683a165c7773b55f2b3d26f2
  17. 683a33037773b55f2b894929
  18. 683db0924fe78cbe3a3fa6e7
  19. 683db7e84fe78cbe3a527ea2
  20. 683e33d54fe78cbe3a9a2742
  21. 683e60a94fe78cbe3a10cb54
  22. 683a149b7773b55f2b387bc4
  23. 683e4e0d4fe78cbe3adf078a
  24. 683da7984fe78cbe3a27d94e
  25. 683e94294fe78cbe3a9a00d6
  26. 683e75d44fe78cbe3a49257a
  27. 683e8dc94fe78cbe3a88f376
  28. 683e0cff4fe78cbe3a32e543
  29. 683db1774fe78cbe3a41fa8b
  30. 683e1a444fe78cbe3a5646a4

📊 测试集数据规模: 51141 行


# 每次读取10场并处理，直至读完

In [ ]:
import os
import gc
import json
import numpy as np
import polars as pl
from scipy.spatial import ConvexHull, Delaunay
from scipy.ndimage import correlate
# 如果你仍然需要原始的 KMeans，请取消下行注释
# from sklearn.cluster import KMeans 

# ==============================================================================
# 1. 全局配置、物理常量与先验网格 (初始化一次，永久驻留内存)
# ==============================================================================
OUTPUT_DIR = "./processed_as_team"  # 请替换为实际路径
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 物理与球场常量
PITCH_LENGTH = 105.0
PITCH_WIDTH = 68.0
GOAL_X = 105.0
GOAL_Y_CENTER = 34.0
GOAL_POST_TOP = 37.66
GOAL_POST_BOT = 30.34
GOAL_WIDTH = 7.32

# 动力学参数
MAX_SPEED = 5.0
MAX_ACCEL = 7.0
LAMBDA_ATT = 3.99
KAPPA = 1.72
LAMBDA_DEF = LAMBDA_ATT * KAPPA
S_PARAM = 0.54  

# 初始化全场 1x1 米稠密网格矩阵 (Shape: 68, 105)
x_bins = np.arange(0, int(PITCH_LENGTH), 1)
y_bins = np.arange(0, int(PITCH_WIDTH), 1)
X_GRID, Y_GRID = np.meshgrid(x_bins, y_bins)

# 计算静态 68x105 网格的距离矩阵
STATIC_GRID_DIST = np.hypot(GOAL_X - X_GRID, GOAL_Y_CENTER - Y_GRID)

# 计算静态 68x105 网格的可视张角矩阵
_numerator = GOAL_WIDTH * (GOAL_X - X_GRID)
_denominator = (GOAL_X - X_GRID)**2 + (Y_GRID - GOAL_POST_TOP) * (Y_GRID - GOAL_POST_BOT)
STATIC_GRID_ANGLE = np.abs(np.arctan2(_numerator, _denominator))
STATIC_GRID_ANGLE[X_GRID >= GOAL_X] = 0.0  # 修正底线后的异常值

# 全局：构建 8 个区域的空间核
MAX_DIST = 15
KERNEL_SIZE = 2 * MAX_DIST + 1
CENTER = MAX_DIST

y_k, x_k = np.ogrid[0:KERNEL_SIZE, 0:KERNEL_SIZE]
dx_k = x_k - CENTER
dy_k = y_k - CENTER
dist_k = np.hypot(dx_k, dy_k)

mask_0_5 = (dist_k <= 5.0)
mask_5_15 = (dist_k > 5.0) & (dist_k <= 15.0)
mask_lf = (dx_k >= 0) & (dy_k >= 0)
mask_rf = (dx_k >= 0) & (dy_k < 0)
mask_lb = (dx_k < 0) & (dy_k >= 0)
mask_rb = (dx_k < 0) & (dy_k < 0)

PC_KERNELS = {
    "grid_pc_5m_lf": (mask_0_5 & mask_lf).astype(float),
    "grid_pc_5m_rf": (mask_0_5 & mask_rf).astype(float),
    "grid_pc_5m_lb": (mask_0_5 & mask_lb).astype(float),
    "grid_pc_5m_rb": (mask_0_5 & mask_rb).astype(float),
    "grid_pc_15m_lf": (mask_5_15 & mask_lf).astype(float),
    "grid_pc_15m_rf": (mask_5_15 & mask_rf).astype(float),
    "grid_pc_15m_lb": (mask_5_15 & mask_lb).astype(float),
    "grid_pc_15m_rb": (mask_5_15 & mask_rb).astype(float),
}

# Karun Singh 12x8 xT 权重矩阵
XT_MATRIX = np.array([
    [0.003, 0.004, 0.003, 0.004, 0.004, 0.005, 0.007, 0.008, 0.01, 0.013, 0.018, 0.025, 0.037, 0.048, 0.062, 0.066],
    [0.003, 0.005, 0.004, 0.004, 0.004, 0.005, 0.007, 0.008, 0.011, 0.015, 0.020, 0.029, 0.042, 0.056, 0.075, 0.092],
    [0.003, 0.003, 0.003, 0.004, 0.004, 0.006, 0.007, 0.009, 0.012, 0.016, 0.023, 0.034, 0.046, 0.067, 0.094, 0.103],
    [0.004, 0.004, 0.004, 0.004, 0.004, 0.006, 0.007, 0.009, 0.012, 0.016, 0.023, 0.034, 0.044, 0.068, 0.089, 0.123],
    [0.005, 0.005, 0.006, 0.004, 0.004, 0.006, 0.007, 0.01, 0.013, 0.017, 0.025, 0.033, 0.0480, 0.068, 0.126, 0.158],
    [0.004, 0.005, 0.007, 0.004, 0.004, 0.006, 0.008, 0.009, 0.013, 0.017, 0.025, 0.033, 0.052, 0.089, 0.171, 0.413],
    [0.004, 0.005, 0.007, 0.005, 0.005, 0.005, 0.008, 0.01, 0.012, 0.019, 0.025, 0.035, 0.042, 0.116, 0.176, 0.371],
    [0.005, 0.006, 0.007, 0.005, 0.005, 0.006, 0.008, 0.009, 0.014, 0.018, 0.024, 0.037, 0.046, 0.077, 0.124, 0.158],
    [0.004, 0.005, 0.005, 0.004, 0.004, 0.006, 0.008, 0.009, 0.013, 0.017, 0.024, 0.036, 0.047, 0.06, 0.096, 0.126],
    [0.003, 0.004, 0.004, 0.004, 0.005, 0.006, 0.008, 0.01, 0.013, 0.016, 0.024, 0.036, 0.049, 0.068, 0.096, 0.136],
    [0.003, 0.003, 0.004, 0.004, 0.005, 0.006, 0.008, 0.01, 0.012, 0.015, 0.022, 0.031, 0.044, 0.060, 0.077, 0.096],
    [0.002, 0.003, 0.004, 0.004, 0.005, 0.006, 0.007, 0.008, 0.011, 0.014, 0.019, 0.026, 0.039, 0.051, 0.063, 0.068]
])


# ==============================================================================
# 2. Polars Schema 与全局状态定义
# ==============================================================================
PC_DTYPE = pl.Struct([
    pl.Field("pc_5m_lf", pl.Float64), pl.Field("pc_5m_rf", pl.Float64),
    pl.Field("pc_5m_lb", pl.Float64), pl.Field("pc_5m_rb", pl.Float64),
    pl.Field("pc_15m_lf", pl.Float64), pl.Field("pc_15m_rf", pl.Float64),
    pl.Field("pc_15m_lb", pl.Float64), pl.Field("pc_15m_rb", pl.Float64),

    # ---> 新增：传球维度与目标区域特征 <---
    pl.Field("pass_length", pl.Float64),
    pl.Field("dest_pc_5m", pl.Float64),

    # ---> 新增：全场网格相对 Origin 的几何特征 <---
    pl.Field("grid_dist_to_origin", pl.List(pl.Float32)),
    pl.Field("grid_angle_to_origin", pl.List(pl.Float32)),
    
    pl.Field("pos_formation_height", pl.Float64), pl.Field("pos_formation_width", pl.Float64),
    pl.Field("def_formation_height", pl.Float64), pl.Field("def_formation_width", pl.Float64),
    pl.Field("pos_space_0_area", pl.Float64), pl.Field("pos_space_1_area", pl.Float64),
    pl.Field("pos_space_2_area", pl.Float64), pl.Field("pos_space_3_area", pl.Float64),
    pl.Field("def_space_0_area", pl.Float64), pl.Field("def_space_1_area", pl.Float64),
    pl.Field("def_space_2_area", pl.Float64), pl.Field("def_space_3_area", pl.Float64),
    
    pl.Field("pc_in_def_space_0", pl.Float64), pl.Field("pc_in_def_space_1", pl.Float64),
    pl.Field("pc_in_def_space_2", pl.Float64), pl.Field("pc_in_def_space_3", pl.Float64),
    pl.Field("pc_ratio_def_space_0", pl.Float64), pl.Field("pc_ratio_def_space_1", pl.Float64),
    pl.Field("pc_ratio_def_space_2", pl.Float64), pl.Field("pc_ratio_def_space_3", pl.Float64),
    
    pl.Field("dense_pc_grid", pl.List(pl.Float32)),
    pl.Field("dense_owner_team", pl.List(pl.Int8)),
    pl.Field("dense_owner_player", pl.List(pl.Utf8)),
    pl.Field("dense_pos_longitudinal_space", pl.List(pl.Int8)),
    pl.Field("dense_def_longitudinal_space", pl.List(pl.Int8)),

    pl.Field("dist_to_goal", pl.Float64),
    pl.Field("angle_to_goal", pl.Float64),
    pl.Field("grid_dist_to_goal", pl.List(pl.Float32)),
    pl.Field("grid_angle_to_goal", pl.List(pl.Float32)),

    pl.Field("grid_pc_5m_lf", pl.List(pl.Float32)), pl.Field("grid_pc_5m_rf", pl.List(pl.Float32)),
    pl.Field("grid_pc_5m_lb", pl.List(pl.Float32)), pl.Field("grid_pc_5m_rb", pl.List(pl.Float32)),
    pl.Field("grid_pc_15m_lf", pl.List(pl.Float32)), pl.Field("grid_pc_15m_rf", pl.List(pl.Float32)),
    pl.Field("grid_pc_15m_lb", pl.List(pl.Float32)), pl.Field("grid_pc_15m_rb", pl.List(pl.Float32)),

    pl.Field("path_pc_min_1", pl.Float64), pl.Field("path_pc_min_2", pl.Float64), pl.Field("path_pc_min_3", pl.Float64),
    pl.Field("grid_path_pc_min_1", pl.List(pl.Float32)),
    pl.Field("grid_path_pc_min_2", pl.List(pl.Float32)),
    pl.Field("grid_path_pc_min_3", pl.List(pl.Float32)),

    # 在 pc_dtype 的内部追加这两个字段：
    pl.Field("active_player_ids", pl.List(pl.Utf8)),
    pl.Field("grid_to_player_dist_tensor", pl.List(pl.Float32)),
])

# 动态生成完美对齐的空特征字典
GLOBAL_EMPTY_RES = {}
for f in PC_DTYPE.fields:
    if "List" in str(f.dtype):
        GLOBAL_EMPTY_RES[f.name] = []
    elif "Utf8" in str(f.dtype) or "String" in str(f.dtype):
        GLOBAL_EMPTY_RES[f.name] = ""
    else:
        GLOBAL_EMPTY_RES[f.name] = 0.0

HULL_DTYPE = pl.Struct([
    pl.Field("defensive_hull_area", pl.Float64),
    pl.Field("is_ball_inside_def_hull", pl.Boolean)
])


# ==============================================================================
# 3. 核心计算与功能函数 (扁平化提取)
# ==============================================================================

def calculate_xt(x, y):
    """根据归一化坐标计算 xT 值"""
    col = min(max(0, int(x * 16)), 15)
    row = min(max(0, int(y * 12)), 11)
    return XT_MATRIX[row, col]

def calculate_current_threat_polars(df, n_steps=5, threshold=50, alpha=0.87):
    """时序衰减威胁计算"""
    step_xt_cols = []
    is_continuous = pl.lit(True)
    
    for i in range(1, n_steps + 1):
        current_start = pl.col('start_frame').shift(-i)
        prev_end = pl.col('end_frame').shift(-(i - 1))
        current_xt = pl.col('start_xT').shift(-i)
        
        gap = current_start - prev_end
        is_continuous = is_continuous & (gap >= 0) & (gap <= threshold)
        
        step_xt = pl.when(is_continuous & current_xt.is_not_null()) \
                    .then(current_xt * (alpha ** i)) \
                    .otherwise(0.0)
        step_xt_cols.append(step_xt)
    
    return df.with_columns(
        pl.max_horizontal(step_xt_cols).alias('current_threat')
    )

def calculate_defensive_hull_features(struct_row) -> dict:
    """计算防守方凸包特征"""
    empty_res = {"defensive_hull_area": 0.0, "is_ball_inside_def_hull": False}
    try:
        bx_norm = struct_row.get("norm_origin_pos_x")
        by_norm = struct_row.get("norm_origin_pos_y")
        
        if bx_norm is None or np.isnan(bx_norm) or by_norm is None or np.isnan(by_norm):
            return empty_res
            
        bx, by = float(bx_norm) * PITCH_LENGTH, float(by_norm) * PITCH_WIDTH

        px = json.loads(struct_row["stat_posx_array"].decode('utf-8'))
        py = json.loads(struct_row["stat_posy_array"].decode('utf-8'))
        is_teammate = json.loads(struct_row["stat_possteam_array"].decode('utf-8'))
        
        if "stat_is_gk_array" in struct_row and struct_row["stat_is_gk_array"] is not None:
            is_gk = json.loads(struct_row["stat_is_gk_array"].decode('utf-8'))
        else:
            is_gk = [False] * len(px)
        
        def_x, def_y = [], []
        for i in range(len(px)):
            if px[i] is None or py[i] is None: continue
            if not is_teammate[i] and not is_gk[i]:
                def_x.append(px[i] * PITCH_LENGTH)
                def_y.append(py[i] * PITCH_WIDTH)
                
        if len(def_x) < 3:
            return empty_res
            
        pts = np.column_stack((def_x, def_y))
        hull = ConvexHull(pts)
        area = hull.volume  
        
        hull_delaunay = Delaunay(pts[hull.vertices])
        is_inside = hull_delaunay.find_simplex((bx, by)) >= 0
        
        return {"defensive_hull_area": float(area), "is_ball_inside_def_hull": bool(is_inside)}
    except Exception:
        return empty_res

def get_longitudinal_spaces_fast(px_array, x_grid, n_lines=3):
    """极速版 1D 阵型聚类"""
    if len(px_array) < n_lines:
        return np.zeros_like(x_grid, dtype=np.int8)
    
    px_sorted = np.sort(px_array)
    if len(px_sorted) >= 11:
        if px_sorted[0] < (PITCH_LENGTH - px_sorted[-1]):
            outfield_x = px_sorted[1:]  
        else:
            outfield_x = px_sorted[:-1] 
    else:
        outfield_x = px_sorted
        
    gaps = np.diff(outfield_x)
    cut_indices = np.argsort(gaps)[-(n_lines - 1):]
    cut_indices = np.sort(cut_indices) + 1  
    
    splits = np.split(outfield_x, cut_indices)
    lines = np.array([np.mean(s) for s in splits if len(s) > 0])
    
    if len(lines) < n_lines:
        lines = np.linspace(outfield_x[0], outfield_x[-1], n_lines)
    else:
        lines = np.sort(lines)
    
    return np.digitize(x_grid, bins=lines).astype(np.int8)

def compute_dense_pc_and_rings(bx, by, dest_x, dest_y, att_pos, att_vel, att_pids, def_pos, def_vel, def_pids):
    """底层 PC 计算与特征工程张量运算核心"""
    N_att, N_def = len(att_pos), len(def_pos)
    N = N_att + N_def
    
    if N_att == 0 or N_def == 0:
        return GLOBAL_EMPTY_RES
        
    pos = np.vstack([att_pos, def_pos])  
    vel = np.vstack([att_vel, def_vel])  
    real_pids_array = np.array(att_pids + def_pids)
    
    lam = np.empty(N)
    lam[:N_att] = LAMBDA_ATT
    lam[N_att:] = LAMBDA_DEF
    
    px = pos[:, 0].reshape(-1, 1, 1)
    py = pos[:, 1].reshape(-1, 1, 1)
    vx = vel[:, 0].reshape(-1, 1, 1)
    vy = vel[:, 1].reshape(-1, 1, 1)
    lam = lam.reshape(-1, 1, 1)

    dx = X_GRID[None, :, :] - px  
    dy = Y_GRID[None, :, :] - py
    dist = np.hypot(dx, dy)
    dist_safe = np.where(dist == 0, 1e-5, dist)
    
    v_proj = (vx * dx + vy * dy) / dist_safe
    v_proj = np.clip(v_proj, -MAX_SPEED, MAX_SPEED)
    
    t_acc = (MAX_SPEED - v_proj) / MAX_ACCEL
    d_acc = v_proj * t_acc + 0.5 * MAX_ACCEL * (t_acc ** 2)
    
    tau = np.where(
        dist < d_acc,
        (-v_proj + np.sqrt(np.clip(v_proj**2 + 2.0 * MAX_ACCEL * dist, 0, None))) / MAX_ACCEL,
        t_acc + (dist - d_acc) / MAX_SPEED
    )
    tau = np.clip(tau, 0, None)  
    tau_min = np.min(tau, axis=0) 
    delta_tau = tau - tau_min[None, :, :] 
    
    active_mask = delta_tau <= 1.2 
    weights = lam * np.exp(-delta_tau / S_PARAM) * active_mask
    
    sum_weights = np.sum(weights, axis=0)
    sum_weights_safe = np.where(sum_weights == 0, 1.0, sum_weights)
    ppcf = weights / sum_weights_safe  
    
    att_ppcf = np.sum(ppcf[:N_att], axis=0)          
    owner_id_internal = np.argmax(ppcf, axis=0)               
    owner_team = (owner_id_internal < N_att).astype(np.int8)  
    real_owner_player_matrix = real_pids_array[owner_id_internal]

    def calc_formation_dims(pos_array):
        if len(pos_array) < 2: return 0.0, 0.0
        p_x, p_y = pos_array[:, 0], pos_array[:, 1]
        px_sorted = np.sort(p_x)
        if len(px_sorted) >= 11:
            gk_x = px_sorted[0] if px_sorted[0] < (PITCH_LENGTH - px_sorted[-1]) else px_sorted[-1]
            mask = p_x != gk_x
            outfield_x, outfield_y = p_x[mask], p_y[mask]
        else:
            outfield_x, outfield_y = p_x, p_y
        if len(outfield_x) < 2: return 0.0, 0.0
        return float(np.ptp(outfield_x)), float(np.ptp(outfield_y))

    pos_height, pos_width = calc_formation_dims(att_pos)
    def_height, def_width = calc_formation_dims(def_pos)

    pos_space_matrix = get_longitudinal_spaces_fast(att_pos[:, 0], X_GRID, n_lines=3)
    def_space_matrix = get_longitudinal_spaces_fast(def_pos[:, 0], X_GRID, n_lines=3)
    
    pos_areas = [float(np.sum(pos_space_matrix == i)) for i in range(4)]
    def_areas = [float(np.sum(def_space_matrix == i)) for i in range(4)]
    
    total_att_pc = float(np.sum(att_ppcf))
    pc_in_def_spaces = [float(np.sum(att_ppcf[def_space_matrix == i])) for i in range(4)]
    pc_ratio_def_spaces = [(val / total_att_pc) if total_att_pc > 0 else 0.0 for val in pc_in_def_spaces]
    
    dx_ball = X_GRID - bx
    dy_ball = Y_GRID - by
    dist_to_ball = np.hypot(dx_ball, dy_ball)
    mask_0_5 = dist_to_ball <= 5.0
    mask_5_15 = (dist_to_ball > 5.0) & (dist_to_ball <= 15.0)
    mask_lf = (dx_ball >= 0) & (dy_ball >= 0)
    mask_rf = (dx_ball >= 0) & (dy_ball < 0)
    mask_lb = (dx_ball < 0) & (dy_ball >= 0)
    mask_rb = (dx_ball < 0) & (dy_ball < 0)
    
    def get_mean_pc(dist_mask, quad_mask):
        valid_cells = att_ppcf[dist_mask & quad_mask]
        return float(np.mean(valid_cells)) if len(valid_cells) > 0 else 0.0
    
    if bx is not None and by is not None and not np.isnan(by):
        d_dist = float(np.hypot(GOAL_X - bx, GOAL_Y_CENTER - by))
        d_num = GOAL_WIDTH * (GOAL_X - bx)
        d_den = (GOAL_X - bx)**2 + (by - GOAL_POST_TOP) * (by - GOAL_POST_BOT)
        d_angle = float(np.abs(np.arctan2(d_num, d_den)))
        if bx >= GOAL_X: d_angle = 0.0
    else:
        d_dist, d_angle = 0.0, 0.0
    
    att_ppcf_2d = att_ppcf.reshape((len(y_bins), len(x_bins)))
    
    # 提取路径脆弱性
    if dest_x is not None and dest_y is not None and not np.isnan(dest_x) and not np.isnan(dest_y):
        dist_to_dest = np.hypot(dest_x - bx, dest_y - by)
        n_pts = max(5, int(dist_to_dest * 2)) 
        t_path = np.linspace(0, 1, n_pts)
        px_path = np.clip(np.round(bx + t_path * (dest_x - bx)).astype(int), 0, 104)
        py_path = np.clip(np.round(by + t_path * (dest_y - by)).astype(int), 0, 67)
        path_pc = att_ppcf_2d[py_path, px_path]
        path_pc_sorted = np.sort(path_pc)
        if len(path_pc_sorted) >= 3:
            pc_min_1, pc_min_2, pc_min_3 = float(path_pc_sorted[0]), float(path_pc_sorted[1]), float(path_pc_sorted[2])
        else:
            pc_min_1 = pc_min_2 = pc_min_3 = float(path_pc_sorted[0])
    else:
        pc_min_1 = pc_min_2 = pc_min_3 = 0.0

    N_STEPS = 40
    t_tensor = np.linspace(0, 1, N_STEPS)[:, None, None]
    ray_x = np.clip(np.round(bx + t_tensor * (X_GRID - bx)).astype(int), 0, 104)
    ray_y = np.clip(np.round(by + t_tensor * (Y_GRID - by)).astype(int), 0, 67)
    ray_pc_vals = att_ppcf_2d[ray_y, ray_x]
    ray_pc_sorted = np.sort(ray_pc_vals, axis=0)[:3, :, :]

    ones_grid = np.ones_like(att_ppcf)
    dense_grid_features = {}
    
    for feature_name, kernel in PC_KERNELS.items():
        pc_sum_grid = correlate(att_ppcf, kernel, mode='constant', cval=0.0)
        area_count_grid = correlate(ones_grid, kernel, mode='constant', cval=0.0)
        area_count_safe = np.where(area_count_grid == 0, 1.0, area_count_grid)
        pc_mean_grid = pc_sum_grid / area_count_safe
        dense_grid_features[feature_name] = np.round(pc_mean_grid.flatten(), 4).tolist()

    # =====================================================================
    # 🚀 新增 1：传球距离与目标区域 (Destination) 5m 控制力
    # =====================================================================
    if dest_x is not None and dest_y is not None and not np.isnan(dest_x) and not np.isnan(dest_y):
        # 1. 计算传球绝对长度
        pass_length = float(np.hypot(dest_x - bx, dest_y - by))
        
        # 2. 计算 Destination 5m 控制力 (基于目标物理坐标生成局部掩膜)
        dx_dest = X_GRID - dest_x
        dy_dest = Y_GRID - dest_y
        dist_to_dest = np.hypot(dx_dest, dy_dest)
        mask_dest_5m = dist_to_dest <= 5.0
        
        valid_dest_cells = att_ppcf[mask_dest_5m]
        dest_pc_5m = float(np.mean(valid_dest_cells)) if len(valid_dest_cells) > 0 else 0.0
    else:
        # 如果不是传球事件（无目的地），默认置零
        pass_length = 0.0
        dest_pc_5m = 0.0

    # =====================================================================
    # 🚀 新增 2：全场 7140 个网格相对起点 (Origin) 的距离与角度
    # =====================================================================
    grid_dx_origin = X_GRID - bx
    grid_dy_origin = Y_GRID - by
    
    # 向量化计算：网格相对起点的距离矩阵
    grid_dist_origin_matrix = np.hypot(grid_dx_origin, grid_dy_origin)
    
    # 向量化计算：网格相对起点的角度矩阵 (使用 arctan2 返回弧度，范围 [-pi, pi])
    grid_angle_origin_matrix = np.arctan2(grid_dy_origin, grid_dx_origin)
        
    return {
        "pc_5m_lf": get_mean_pc(mask_0_5, mask_lf), "pc_5m_rf": get_mean_pc(mask_0_5, mask_rf),
        "pc_5m_lb": get_mean_pc(mask_0_5, mask_lb), "pc_5m_rb": get_mean_pc(mask_0_5, mask_rb),
        "pass_length": pass_length,
        "dest_pc_5m": dest_pc_5m,
        "grid_dist_to_origin": np.round(grid_dist_origin_matrix.flatten(), 4).tolist(),
        "grid_angle_to_origin": np.round(grid_angle_origin_matrix.flatten(), 4).tolist(),

        "pc_15m_lf": get_mean_pc(mask_5_15, mask_lf), "pc_15m_rf": get_mean_pc(mask_5_15, mask_rf),
        "pc_15m_lb": get_mean_pc(mask_5_15, mask_lb), "pc_15m_rb": get_mean_pc(mask_5_15, mask_rb),
        "pos_formation_height": pos_height, "pos_formation_width": pos_width,
        "def_formation_height": def_height, "def_formation_width": def_width,
        "pos_space_0_area": pos_areas[0], "pos_space_1_area": pos_areas[1],
        "pos_space_2_area": pos_areas[2], "pos_space_3_area": pos_areas[3],
        "def_space_0_area": def_areas[0], "def_space_1_area": def_areas[1],
        "def_space_2_area": def_areas[2], "def_space_3_area": def_areas[3],
        "pc_in_def_space_0": pc_in_def_spaces[0], "pc_in_def_space_1": pc_in_def_spaces[1],
        "pc_in_def_space_2": pc_in_def_spaces[2], "pc_in_def_space_3": pc_in_def_spaces[3],
        "pc_ratio_def_space_0": pc_ratio_def_spaces[0], "pc_ratio_def_space_1": pc_ratio_def_spaces[1],
        "pc_ratio_def_space_2": pc_ratio_def_spaces[2], "pc_ratio_def_space_3": pc_ratio_def_spaces[3],
        "dense_pc_grid": np.round(att_ppcf.flatten(), 4).tolist(),
        "dense_owner_team": owner_team.flatten().tolist(),
        "dense_owner_player": real_owner_player_matrix.flatten().tolist(),
        "dense_pos_longitudinal_space": pos_space_matrix.flatten().tolist(),
        "dense_def_longitudinal_space": def_space_matrix.flatten().tolist(),
        "grid_dist_to_goal": np.round(STATIC_GRID_DIST.flatten(), 4).tolist(),    # 改用全局预计算的矩阵
        "grid_angle_to_goal": np.round(STATIC_GRID_ANGLE.flatten(), 4).tolist(),  # 改用全局预计算的矩阵
        "dist_to_goal": d_dist,
        "angle_to_goal": d_angle,
        "grid_pc_5m_lf": dense_grid_features["grid_pc_5m_lf"],
        "grid_pc_5m_rf": dense_grid_features["grid_pc_5m_rf"],
        "grid_pc_5m_lb": dense_grid_features["grid_pc_5m_lb"],
        "grid_pc_5m_rb": dense_grid_features["grid_pc_5m_rb"],
        "grid_pc_15m_lf": dense_grid_features["grid_pc_15m_lf"],
        "grid_pc_15m_rf": dense_grid_features["grid_pc_15m_rf"],
        "grid_pc_15m_lb": dense_grid_features["grid_pc_15m_lb"],
        "grid_pc_15m_rb": dense_grid_features["grid_pc_15m_rb"],
        "path_pc_min_1": pc_min_1, "path_pc_min_2": pc_min_2, "path_pc_min_3": pc_min_3,
        "grid_path_pc_min_1": np.round(ray_pc_sorted[0].flatten(), 4).tolist(),
        "grid_path_pc_min_2": np.round(ray_pc_sorted[1].flatten(), 4).tolist(),
        "grid_path_pc_min_3": np.round(ray_pc_sorted[2].flatten(), 4).tolist(),
        # 新增：当前帧场上所有有效球员的真实 ID 序列
        "active_player_ids": real_pids_array.tolist(),
        
        # 新增：距离张量展平 (维度为 N * 68 * 105)
        # 读取端还原方法：np.array(row).reshape(len(active_ids), 68, 105)
        "grid_to_player_dist_tensor": np.round(dist, 4).flatten().tolist(),
    }

def apply_unified_pc_calculation(struct_row) -> dict:
    """行级映射：解析 JSON 并分发给计算核"""
    bx_norm = struct_row["norm_origin_pos_x"]
    by_norm = struct_row["norm_origin_pos_y"]
    dest_x_norm = struct_row.get("norm_destination_pos_x")
    dest_y_norm = struct_row.get("norm_destination_pos_y")
    
    if bx_norm is None or np.isnan(bx_norm) or by_norm is None or np.isnan(by_norm):
        return GLOBAL_EMPTY_RES

    dest_x = dest_x_norm * PITCH_LENGTH if dest_x_norm is not None else None
    dest_y = dest_y_norm * PITCH_WIDTH if dest_y_norm is not None else None
    bx, by = bx_norm * PITCH_LENGTH, by_norm * PITCH_WIDTH

    try:
        px = json.loads(struct_row["stat_posx_array"].decode('utf-8'))
        py = json.loads(struct_row["stat_posy_array"].decode('utf-8'))
        vx = json.loads(struct_row["stat_velx_array"].decode('utf-8'))
        vy = json.loads(struct_row["stat_vely_array"].decode('utf-8'))
        is_teammate = json.loads(struct_row["stat_possteam_array"].decode('utf-8'))
        pids = json.loads(struct_row["stat_player_id_array"].decode('utf-8'))
        
        att_pos, att_vel, att_pids = [], [], []
        def_pos, def_vel, def_pids = [], [], []
        
        for i in range(len(px)):
            if px[i] is None or py[i] is None: continue
            
            x, y = px[i] * PITCH_LENGTH, py[i] * PITCH_WIDTH
            v_x = (vx[i] * PITCH_LENGTH) if vx[i] is not None else 0.0
            v_y = (vy[i] * PITCH_WIDTH)  if vy[i] is not None else 0.0
            pid = pids[i]
            
            if is_teammate[i]:
                att_pos.append([x, y])
                att_vel.append([v_x, v_y])
                att_pids.append(pid)
            else:
                def_pos.append([x, y])
                def_vel.append([v_x, v_y])
                def_pids.append(pid)
                
        att_pos = np.array(att_pos) if att_pos else np.empty((0, 2))
        att_vel = np.array(att_vel) if att_vel else np.empty((0, 2))
        def_pos = np.array(def_pos) if def_pos else np.empty((0, 2))
        def_vel = np.array(def_vel) if def_vel else np.empty((0, 2))
        
        return compute_dense_pc_and_rings(bx, by, dest_x, dest_y, att_pos, att_vel, att_pids, def_pos, def_vel, def_pids)
    except Exception:
        return GLOBAL_EMPTY_RES

# ==============================================================================
# 4. 主 Pipeline (结构变得极其清晰)
# ==============================================================================
import os
import polars as pl

def process_and_save_batch(df: pl.DataFrame, team_name: str, batch_idx: int):
    """
    接收 某支球队某一特定批次 的 DataFrame，通过纯 Pipeline 调用外部模块执行计算。
    """
    # 1. 修正变量名错误：将 batch_index 改为 batch_idx，batch_df 改为 df
    print(f"🔄 开始处理 [{team_name}] 批次 {batch_idx}，共 {df.shape[0]} 行追踪数据...")
    df = df.sort(["match_id", "team", "start_frame"])

    # 1. 计算 xT (Expected Threat)
    df = df.with_columns(
        pl.struct(["norm_origin_pos_x", "norm_origin_pos_y"])
        .map_elements(
            lambda r: calculate_xt(r["norm_origin_pos_x"], r["norm_origin_pos_y"]),
            return_dtype=pl.Float64
        ).alias("start_xT")
    )

    # 2. 时序威胁衰减
    df = calculate_current_threat_polars(df, n_steps=5)

    # 3. 计算防守方凸包面积
    required_hull_cols = [
        "norm_origin_pos_x", "norm_origin_pos_y", 
        "stat_posx_array", "stat_posy_array", "stat_possteam_array",
    ]
    df = df.with_columns(
        pl.struct(required_hull_cols)
        .map_elements(calculate_defensive_hull_features, return_dtype=HULL_DTYPE)
        .alias("defensive_shape_features")
    ).unnest("defensive_shape_features")

    # 4. 统一战术特征提取 (PC / 阵型 / 路径)
    required_pc_cols = [
        "norm_destination_pos_x", "norm_destination_pos_y",
        "norm_origin_pos_x", "norm_origin_pos_y", "stat_posx_array", "stat_posy_array", 
        "stat_velx_array", "stat_vely_array", "stat_possteam_array", "stat_player_id_array" 
    ]
    
    new_col_names = [field.name for field in PC_DTYPE.fields]
    cols_to_drop = [col for col in new_col_names if col in df.columns]
    if cols_to_drop:
        df = df.drop(cols_to_drop)

    df = df.with_columns(
        pl.struct(required_pc_cols)
        .map_elements(apply_unified_pc_calculation, return_dtype=PC_DTYPE)
        .alias("unified_tactical_features")
    ).unnest("unified_tactical_features")

    # 5. 落盘保存 (防覆盖安全优化)
    # 提取安全的球队名称字符串，去除可能导致路径解析失败的空格或斜杠
    if isinstance(team_name, tuple):
        team_str = str(team_name[0])
    else:
        team_str = str(team_name)
    safe_team_name = team_str.replace(" ", "_").replace("/", "_").strip()
    
    # 将球队名和三位补零的批次号组合成唯一文件名
    file_name = f"{safe_team_name}_batch_{batch_idx:03d}.parquet"
    save_path = os.path.join(OUTPUT_DIR, file_name)
    
    # 确保保存目录存在
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    df.write_parquet(save_path)
    print(f"✅ 球队 [{team_str}] 批次 {batch_idx} 处理完毕，已保存至: {save_path}")
"""
# ==============================================================================
# 5. 批处理调度引擎
# ==============================================================================
def run_batch_processing(match_dictionary: dict, batch_size: int = 10):
    all_keys = list(match_dictionary.keys())
    total_matches = len(all_keys)
    
    print(f"总计发现 {total_matches} 场比赛。配置：每 {batch_size} 场保存为一个批次。")
    batches = [all_keys[i : i + batch_size] for i in range(0, total_matches, batch_size)]
    
    for batch_idx, batch_keys in enumerate(batches, start=1):
        print(f"\n" + "="*50)
        print(f"▶️ 正在提取批次 {batch_idx}/{len(batches)} (包含 {len(batch_keys)} 场比赛)")
        
        current_batch_df = pl.concat([match_dictionary[k] for k in batch_keys])
        process_and_save_batch(current_batch_df, batch_idx)
        
        del current_batch_df
        gc.collect()
        
    print("\n🎉 所有批次处理完成！可以安全进入机器学习建模阶段。")

# 启动引擎 (此处 match_dict 需由外部传入)
run_batch_processing(match_dict, batch_size=10)
"""

'\n# ==============================================================================\n# 5. 批处理调度引擎\n# ==============================================================================\ndef run_batch_processing(match_dictionary: dict, batch_size: int = 10):\n    all_keys = list(match_dictionary.keys())\n    total_matches = len(all_keys)\n\n    print(f"总计发现 {total_matches} 场比赛。配置：每 {batch_size} 场保存为一个批次。")\n    batches = [all_keys[i : i + batch_size] for i in range(0, total_matches, batch_size)]\n\n    for batch_idx, batch_keys in enumerate(batches, start=1):\n        print(f"\n" + "="*50)\n        print(f"▶️ 正在提取批次 {batch_idx}/{len(batches)} (包含 {len(batch_keys)} 场比赛)")\n\n        current_batch_df = pl.concat([match_dictionary[k] for k in batch_keys])\n        process_and_save_batch(current_batch_df, batch_idx)\n\n        del current_batch_df\n        gc.collect()\n\n    print("\n🎉 所有批次处理完成！可以安全进入机器学习建模阶段。")\n\n# 启动引擎 (此处 match_dict 需由外部传入)\nrun_batch_processing(match_dict, batch_size=10)\n

: 

In [ ]:
def run_batch_processing_by_team(df: pl.DataFrame, team_col: str = "team", batch_size: int = 19):
    """
    先根据球队分组，然后在每支球队内部，按每 19 场比赛划分为一个批次进行处理。
    """
    # 1. 按照球队进行顶层划分 (返回 {team_name: team_df} 的字典)
    teams_dict = df.partition_by(team_col, as_dict=True)
    total_teams = len(teams_dict)
    
    print(f"总计发现 {total_teams} 支球队。配置：每支球队每 {batch_size} 场保存为一个批次。\n")

    for team_name, team_df in teams_dict.items():
        print("="*50)
        print(f"🛡️ 正在处理球队: {team_name}")
        
        # 2. 提取该队的所有比赛，并对每一场比赛的数据按帧排序
        match_dict = {
            match_id: sub_df.sort("start_frame")
            for match_id, sub_df in team_df.partition_by("match_id", as_dict=True).items()
        }
        
        # 获取该队所有的比赛 ID
        all_match_keys = list(match_dict.keys())
        total_matches = len(all_match_keys)
        
        print(f"  📊 发现该队共有 {total_matches} 场比赛。")
        
        # 3. 将比赛划分为 19 场一个 batch
        batches = [all_match_keys[i : i + batch_size] for i in range(0, total_matches, batch_size)]
        
        for batch_idx, batch_keys in enumerate(batches, start=1):
            print(f"  ▶️ 提取批次 {batch_idx}/{len(batches)} (包含 {len(batch_keys)} 场比赛)")
            
            # 合并当前批次的 DataFrame
            current_batch_df = pl.concat([match_dict[k] for k in batch_keys])
            
            # 4. 传入下游处理与保存函数
            # 注意：此处必须传入 team_name，否则不同球队的同一批次（如 batch_1）会互相覆盖文件
            process_and_save_batch(current_batch_df, team_name, batch_idx)
            
            # 释放当前 batch 占用的内存
            del current_batch_df
            gc.collect()
            
        # 释放该球队层级的字典内存
        del match_dict
        gc.collect()
        
    print("\n🎉 所有球队批次处理完成！可以安全进入机器学习建模阶段。")

# 启动引擎 (直接传入包含所有数据的全量 DataFrame)
run_batch_processing_by_team(df_total, team_col="team", batch_size=5)

总计发现 20 支球队。配置：每支球队每 5 场保存为一个批次。

🛡️ 正在处理球队: (407,)
  📊 发现该队共有 38 场比赛。
  ▶️ 提取批次 1/8 (包含 5 场比赛)
🔄 开始处理 [(407,)] 批次 1，共 4297 行追踪数据...
✅ 球队 [407] 批次 1 处理完毕，已保存至: ./processed_as_team/407_batch_001.parquet
  ▶️ 提取批次 2/8 (包含 5 场比赛)
🔄 开始处理 [(407,)] 批次 2，共 4788 行追踪数据...
✅ 球队 [407] 批次 2 处理完毕，已保存至: ./processed_as_team/407_batch_002.parquet
  ▶️ 提取批次 3/8 (包含 5 场比赛)
🔄 开始处理 [(407,)] 批次 3，共 3770 行追踪数据...
✅ 球队 [407] 批次 3 处理完毕，已保存至: ./processed_as_team/407_batch_003.parquet
  ▶️ 提取批次 4/8 (包含 5 场比赛)
🔄 开始处理 [(407,)] 批次 4，共 3477 行追踪数据...
✅ 球队 [407] 批次 4 处理完毕，已保存至: ./processed_as_team/407_batch_004.parquet
  ▶️ 提取批次 5/8 (包含 5 场比赛)
🔄 开始处理 [(407,)] 批次 5，共 3865 行追踪数据...
✅ 球队 [407] 批次 5 处理完毕，已保存至: ./processed_as_team/407_batch_005.parquet
  ▶️ 提取批次 6/8 (包含 5 场比赛)
🔄 开始处理 [(407,)] 批次 6，共 3121 行追踪数据...
✅ 球队 [407] 批次 6 处理完毕，已保存至: ./processed_as_team/407_batch_006.parquet
  ▶️ 提取批次 7/8 (包含 5 场比赛)
🔄 开始处理 [(407,)] 批次 7，共 3691 行追踪数据...
✅ 球队 [407] 批次 7 处理完毕，已保存至: ./processed_as_team/407_batch_007.parquet
  ▶️ 提取批次 8/